# Tests — parameter estimation (LMC model)

Validation of the **direct LMC** spatial model (the `\new{}` model in [main.tex](Spatialisation-Rain-Occurrence-Generator-article/main.tex)): three latent fields $W^c, W^{(0)}, W^{(1)}$, mixing weight $\lambda$, parameter $\theta=(\lambda,\sigma_{w^c},\sigma_{w^{(0)}},\sigma_{w^{(1)}})$ — and of its **state-dependent-$\lambda^{(r)}$ variant** (the blue "alternative suggestion" in main.tex, $\theta=(\lambda^{(0)},\lambda^{(1)},\sigma_{w^c},\sigma_{w^{(0)}},\sigma_{w^{(1)}})$). The single-latent-field model and its tests are archived under [archive_old_single_latent_field/](archive_old_single_latent_field/).

We check that the **vectorized `approx_cdf` pairwise MLE** recovers a known $\theta$, for both models:

1. **Smoke runs** — short $\theta$-recovery experiments via `run_simulation_mle_experiment` / `mle_theta_pairwise`, for the shared-$\lambda$ model (1.a) and the state-dependent-$\lambda^{(r)}$ variant (1.b). Results are written to `experiment_outputs/` to verify that saving to disk works before launching the long runs.
2. **Consistency in `nb_steps`** — long runs (many seeds, nested history lengths) comparing the estimation error of both models as the history grows. Every fit is appended to a CSV on disk as soon as it finishes, so the experiment survives lost connections / session timeouts (e.g. on GitHub Codespaces) and resumes where it stopped.

Histories are drawn with the Cholesky simulator `simulate_cholesky` (three latent factors, with burn-in). The `approx_cdf` kernel itself is derived in [approx_cdf_adaptation/approx_cdf_explained.ipynb](approx_cdf_adaptation/approx_cdf_explained.ipynb).

## Setup

Build `dict_model_params` exactly as in the pipeline notebook (imports, single-site fit outputs, station subset, exit-probability closures). These helpers are model-agnostic and shared with the old model.

In [1]:
import sys, pathlib
_HERE = pathlib.Path.cwd().resolve()
_REPO_ROOT = next(
    (p for p in (_HERE, *_HERE.parents) if (p / "article_code").is_dir()),
    None,
)
if _REPO_ROOT is None:
    raise RuntimeError(f"Could not locate repo root containing 'article_code/' from {_HERE}")
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from article_code.util_files import config
from spatial_bmcd.spatial_model import (
    prepare_station_fit_table,
    build_dict_model_params,
    run_simulation_mle_experiment,
    simulate_cholesky,
    mle_theta_pairwise,
    pairwise_loglik_given_theta,
    build_lmc_blocks,
    build_C_from_sigma,
    normalize_theta,
    THETA_BOUNDS_STATE_DEP,
    THETA_X0_STATE_DEP,
)
from spatial_bmcd.spatial_plotting import plot_stations_geomap

In [2]:
# Load single-site outputs produced by notebook 01.
base_filename = (
    f"ecad_data_south_europe_filtered_after_{config.START_YEAR}"
    f"_wet_day_thresh_{config.WET_DAY_THRESHOLD}.json"
)
with open(config.EXPORTS_JSON_DIR / base_filename) as fh:
    spells = json.load(fh)

stations_metadata = pd.read_csv(config.STATION_METADATA_CSV)
stations_used = stations_metadata[stations_metadata["city"].isin(spells.keys())]

fit_folder = (
    config.RESULTS_FIT_DIR
    / f"fit_south_europe_subset_excess_over_{config.WET_DAY_THRESHOLD}"
)
df_fit_dry = pd.read_csv(fit_folder / "dry_spell_fit_egpd1_excess_over_1result_fit_parameters.csv")
df_fit_wet = pd.read_csv(fit_folder / "wet_spell_fit_mixt_geomresult_fit_parameters.csv")
df_fit_dry["city"] = df_fit_dry["data_source"].map(lambda s: s.split()[0])
df_fit_dry["season"] = df_fit_dry["data_source"].map(lambda s: s.split()[-1])

print(f"{len(spells)} stations in spells JSON; {len(stations_used)} in metadata")

209 stations in spells JSON; 209 in metadata


In [3]:
COUNTRIES = ["PT", "ES"]
SEASON = "spring"

df_fit_dry_sub, df_fit_wet_sub = prepare_station_fit_table(
    df_fit_dry, df_fit_wet, stations_used,
    countries=COUNTRIES, season=SEASON,
)
print(f"{len(df_fit_dry_sub)} stations kept for season={SEASON}, countries={COUNTRIES}")

dict_model_params = build_dict_model_params(
    df_fit_dry_sub, df_fit_wet_sub, spells, season=SEASON,
)
print(f"Built model parameters for {len(dict_model_params)} stations")

35 stations kept for season=spring, countries=['PT', 'ES']


100%|██████████| 35/35 [00:00<00:00, 224.36it/s]

Built model parameters for 35 stations


## 1. Maximum-likelihood estimation of $\theta$ on simulated data

Simulate spatial histories with a known $\theta$ and recover it via `run_simulation_mle_experiment` (which calls `mle_theta_pairwise`). Each row of the returned DataFrame is one independent simulation; `simulator="cholesky"` (default) factors the three latent covariances once.

The runs in this section are deliberately **short smoke runs** (`nb_stations=10, nb_steps=500, nb_estimations=1`): their purpose is to check that the pipeline runs end-to-end on this machine and that the results are correctly written to `experiment_outputs/` on disk. Recovery is correspondingly noisy; the long runs are in part 2.

### 1.a Original model: shared mixing weight $\lambda$

**Model fitted here: the original shared-$\lambda$ LMC** of [main.tex](Spatialisation-Rain-Occurrence-Generator-article/main.tex) — the dry and wet fields mix a common field $W^{(c)}$ and a state-specific field $W^{(r)}$ with a single mixing weight $\lambda$, Eq. (4) (`lmc_fields_direct`):

$$
Z^{(r)}_n(\mathbf{s})=(-1)^{r}\sqrt{\lambda}\,W^{(c)}_n(\mathbf{s})+\sqrt{1-\lambda}\,W^{(r)}_n(\mathbf{s}),
\qquad r\in\{0,1\},\quad \lambda\in[0,1].
\tag{4}
$$

The parameter is $\theta=(\lambda,\sigma_{w^c},\sigma_{w^{(0)}},\sigma_{w^{(1)}})$.

In [ ]:
THETA_TRUE = (0.6, 0.30, 0.15, 0.45)   # (lambda, sigma_wc, sigma_w0, sigma_w1)

OUTPUT_DIR = _REPO_ROOT / "spatial_bmcd" / "experiment_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Short smoke run: checks the pipeline end-to-end and that results land on disk
# before launching the long consistency runs of part 2.
df = run_simulation_mle_experiment(
    dict_model_params, theta_true=THETA_TRUE,
    nb_stations=10, nb_steps=500, nb_estimations=1,
    vectorized=True, simulator="cholesky",
)
out_csv = OUTPUT_DIR / "mle_smoke_shared_lambda.csv"
df.to_csv(out_csv, index=False)

print(f"theta_true = (lam, s_wc, s_w0, s_w1) = {THETA_TRUE}")
print(df[["i", "lam_hat", "sigma_wc_hat", "sigma_w0_hat", "sigma_w1_hat", "ll_hat"]].round(3).to_string(index=False))
print(f"saved to {out_csv} ({out_csv.stat().st_size} bytes)")

### 1.b Alternative model: state-dependent mixing weights $\lambda^{(r)}$

**Model fitted here: the alternative suggestion of [main.tex](Spatialisation-Rain-Occurrence-Generator-article/main.tex) (blue)** — the same Gaussian mixture as Eq. (4), but the mixing weight now depends on the state $r$: one weight $\lambda^{(0)}$ for the dry field and one weight $\lambda^{(1)}$ for the wet field (`lmc_fields_direct_state_dep`):

$$
Z^{(r)}_n(\mathbf{s})=(-1)^{r}\sqrt{\lambda^{(r)}}\,W^{(c)}_n(\mathbf{s})+\sqrt{1-\lambda^{(r)}}\,W^{(r)}_n(\mathbf{s}),
\qquad r\in\{0,1\},\quad \lambda^{(r)}\in[0,1].
\tag{5}
$$

The parameter is $\theta=(\lambda^{(0)},\lambda^{(1)},\sigma_{w^c},\sigma_{w^{(0)}},\sigma_{w^{(1)}})$. The shared-$\lambda$ model of 1.a is the special case $\lambda^{(0)}=\lambda^{(1)}=\lambda$; note the dry–wet cross-covariance only identifies the product $\sqrt{\lambda^{(0)}\lambda^{(1)}}$, so the 5-D MLE is somewhat noisier. A 5-component `theta_true` selects this variant everywhere in [spatial_model.py](spatial_model.py) (`normalize_theta` dispatches on the length), and `run_simulation_mle_experiment` then automatically fits the 5-D pairwise MLE (`THETA_BOUNDS_STATE_DEP` / `THETA_X0_STATE_DEP`).

In [ ]:
THETA_TRUE_STATE_DEP = (0.35, 0.75, 0.30, 0.15, 0.45)   # (lam0, lam1, sigma_wc, sigma_w0, sigma_w1)

# Distinct lam0 != lam1 so the state-dependent weights are actually exercised.
# Same short smoke run as 1.a: checks the 5-D pipeline and the on-disk saving.
df_sd = run_simulation_mle_experiment(
    dict_model_params, theta_true=THETA_TRUE_STATE_DEP,
    nb_stations=10, nb_steps=500, nb_estimations=1,
    vectorized=True, simulator="cholesky",
)
out_csv_sd = OUTPUT_DIR / "mle_smoke_state_dep_lambda.csv"
df_sd.to_csv(out_csv_sd, index=False)

print(f"theta_true = (lam0, lam1, s_wc, s_w0, s_w1) = {THETA_TRUE_STATE_DEP}")
print(df_sd[["i", "lam0_hat", "lam1_hat", "sigma_wc_hat", "sigma_w0_hat", "sigma_w1_hat", "ll_hat"]].round(3).to_string(index=False))
print(f"saved to {out_csv_sd} ({out_csv_sd.stat().st_size} bytes)")

## 2. Consistency in `nb_steps` — shared-$\lambda$ (4) vs state-dependent-$\lambda^{(r)}$ (5)

The pairwise composite MLE is consistent, so $\hat\theta$ should concentrate around $\theta_{\text{true}}$ as the history grows — for **both** models of part 1. For each model and each history length $T$ we run **20 independent estimations** (seeds $0\ldots 19$): each seed simulates one long history and is re-fitted on nested prefixes (first $T$ steps), so the error decay is visible at fixed randomness, and the two models are compared on the same experimental design.

The experiment is split into two cells so the (slow) MLEs run once and the plot can be re-drawn freely:

- **Run + save** — the expensive cell (2 models $\times$ 20 seeds $\times$ 6 lengths $=240$ vectorized pairwise MLEs; several hours). It is designed for long unattended runs (e.g. GitHub Codespaces):
  - every fit is **appended immediately** to `experiment_outputs/theta_consistency_by_model.csv` (long format: one row per fitted component, with `model, seed, nb_steps, param, theta_true, theta_hat, abs_err, ll_hat`), so a lost connection or session timeout never loses a finished fit;
  - on re-run, the `(model, seed, nb_steps)` triples already present in the CSV are **skipped**, so the cell resumes exactly where it stopped.
- **Plot** — reloads the CSV and draws one panel per model. Re-run it alone (it also works on partial results while the run cell is still going) without redoing any MLE.

**Error metric.** Everything is reported as a **mean percentage error (MPE)** so all components are on the same dimensionless scale and directly comparable across the two models. For a single fit the per-component percentage error is $100\,|\hat\theta_k-\theta_{\text{true},k}|/\theta_{\text{true},k}$; averaging over the seeds gives the MPE plotted for each component, with error bars at $\pm1$ standard deviation across the seeds. The programmatic check at the end averages the per-component percentage errors and requires, for each model, that this aggregate decreases from the shortest to the longest history.

In [ ]:
# --- Run + save: 2 models x 20 seeds x 6 history lengths, appended to disk fit by fit. ---
THETA_TRUE_BY_MODEL = {
    "shared":    (0.6, 0.30, 0.15, 0.45),          # (lam, s_wc, s_w0, s_w1)        -- model (4)
    "state_dep": (0.35, 0.75, 0.30, 0.15, 0.45),   # (lam0, lam1, s_wc, s_w0, s_w1) -- model (5)
}
KEYS_BY_MODEL = {
    "shared":    ["lam", "sigma_wc", "sigma_w0", "sigma_w1"],
    "state_dep": ["lam0", "lam1", "sigma_wc", "sigma_w0", "sigma_w1"],
}
MLE_KWARGS_BY_MODEL = {
    "shared":    {},
    "state_dep": {"bounds": THETA_BOUNDS_STATE_DEP, "x0": THETA_X0_STATE_DEP},
}
NB_STEPS_GRID = [500, 1000, 2000, 4000, 7000, 10000]
SEEDS = list(range(20))                 # 20 estimations per (model, history length)
NB_STATIONS = 8

names_sub = sorted(dict_model_params)[:NB_STATIONS]
params_sub = {c: dict_model_params[c] for c in names_sub}

def _prefix(h, T):
    """Restrict a simulated history to its first T steps (R/D have T+1 rows, S has T)."""
    return {**h, "R": h["R"][:T + 1], "D": h["D"][:T + 1], "S": h["S"][:T]}

OUTPUT_DIR = _REPO_ROOT / "spatial_bmcd" / "experiment_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONSISTENCY_CSV = OUTPUT_DIR / "theta_consistency_by_model.csv"

# Resume support: (model, seed, nb_steps) triples already on disk are not recomputed,
# so this cell can be interrupted (timeout, lost connection) and simply re-run.
if CONSISTENCY_CSV.exists():
    done = set(map(tuple, pd.read_csv(CONSISTENCY_CSV)[["model", "seed", "nb_steps"]]
                   .drop_duplicates().itertuples(index=False)))
else:
    done = set()

todo = [(m, s, T)
        for m in THETA_TRUE_BY_MODEL for s in SEEDS for T in NB_STEPS_GRID
        if (m, s, T) not in done]
print(f"{len(done)} fits already in {CONSISTENCY_CSV.name} (skipped); {len(todo)} to run")

# For each (model, seed): simulate one long history once, re-fit on nested prefixes.
# Only the current history is kept in memory (todo is ordered by model then seed).
hist_key, hist = None, None
for model, seed, T in tqdm(todo):
    if hist_key != (model, seed):
        hist_key = (model, seed)
        hist = simulate_cholesky(
            THETA_TRUE_BY_MODEL[model], params_sub,
            n_steps=max(NB_STEPS_GRID), n_burn=200, seed=seed,
        )
    th_true = normalize_theta(THETA_TRUE_BY_MODEL[model])
    fit = mle_theta_pairwise(
        _prefix(hist, T), params_sub,
        vectorized=True, **MLE_KWARGS_BY_MODEL[model],
    )
    th = fit["theta_hat"]
    rows = pd.DataFrame([
        {"model": model, "seed": seed, "nb_steps": T, "param": k,
         "theta_true": th_true[k], "theta_hat": th[k],
         "abs_err": abs(th[k] - th_true[k]), "ll_hat": fit["ll_hat"]}
        for k in KEYS_BY_MODEL[model]
    ])
    # Append immediately: a finished fit is on disk before the next one starts.
    rows.to_csv(CONSISTENCY_CSV, mode="a", header=not CONSISTENCY_CSV.exists(), index=False)

print(f"done: all results in {CONSISTENCY_CSV}")

In [ ]:
# --- Plot only (re-runnable without the experiment above, even on partial results): loads the saved CSV. ---
CONSISTENCY_CSV = _REPO_ROOT / "spatial_bmcd" / "experiment_outputs" / "theta_consistency_by_model.csv"
res = pd.read_csv(CONSISTENCY_CSV)

# Percentage error puts every component (and both models) on the same dimensionless
# scale: pct_err = 100 * |theta_hat - theta_true| / theta_true.
res["pct_err"] = 100.0 * res["abs_err"] / res["theta_true"]

MODEL_LABELS = {
    "shared":    r"shared $\lambda$ — model (4)",
    "state_dep": r"state-dependent $\lambda^{(r)}$ — model (5)",
}

fig, axes = plt.subplots(1, len(MODEL_LABELS), figsize=(12.5, 4.2), sharey=True)
agg_by_model = {}
for ax, (model, label) in zip(np.atleast_1d(axes), MODEL_LABELS.items()):
    sub = res[res["model"] == model]
    if sub.empty:
        ax.set_title(f"{label}\n(no results yet)")
        continue
    mpe     = sub.pivot_table(index="nb_steps", columns="param", values="pct_err", aggfunc="mean")
    mpe_std = sub.pivot_table(index="nb_steps", columns="param", values="pct_err", aggfunc="std")
    n_seeds = sub.groupby("nb_steps")["seed"].nunique()

    print(f"\n{label} -- theta_true per component, seeds per T: {dict(n_seeds)}")
    print(mpe.round(2).rename(columns=lambda k: f"MPE%_{k}").to_string())

    for k in mpe.columns:
        ax.errorbar(mpe.index, mpe[k], yerr=mpe_std[k], fmt="o-", capsize=3, label=k)
    ax.set_xlabel("nb_steps $T$")
    ax.set_title(label)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

    # Aggregate MPE (programmatic check): mean over components per fit, then over seeds.
    agg = (sub.groupby(["nb_steps", "seed"])["pct_err"].mean()
              .groupby("nb_steps").mean())
    agg_by_model[model] = agg

np.atleast_1d(axes)[0].set_ylabel(
    r"mean percentage error $100\,|\hat\theta-\theta_{\mathrm{true}}|/\theta_{\mathrm{true}}$ (%)")
fig.suptitle(r"$\hat\theta$ percentage error vs history length ($\pm1$ std over seeds)")
plt.tight_layout()
plt.show()

for model, agg in agg_by_model.items():
    assert agg.iloc[-1] < agg.iloc[0], \
        f"{model}: estimation error should decrease from the shortest to the longest history"
    print(f"OK [{model}]: aggregate MPE decreases "
          f"({agg.iloc[0]:.1f}% at T={agg.index[0]} -> {agg.iloc[-1]:.1f}% at T={agg.index[-1]})")